# arcedge — joint FTTH chain performance benchmark (Colab)

Benchmarks the **joint-aware facility chain** (address → terminal ≤12 →
FDH ≤512 → OLT ≤4000 with measured-cost feedback rounds) on full-SF
Overture data and produces a paste-ready performance report.

**Honest note on GPU:** the design-chain solver is CPU-parallel — the CUDA
backend accelerates the *MCF* solver, not design. What a Colab **GPU
runtime** buys this benchmark is its beefier CPU allocation (GPU-class VMs
often have many more vCPUs than the standard runtime). Machine specs are
captured next to the timings so runs are comparable. For GPU kernel
numbers, use `arcedge_s2m1_benchmark.ipynb`.

What you get: per-round cost + wall-clock, per-tier solver times, final
design totals, `joint_bench.json`, per-tier GeoParquet, and a report block
to paste back.

In [ ]:
# --- parameters -----------------------------------------------------------
TERMINAL_COST = 500        # per terminal (<= TERMINAL_CAP addresses)
TERMINAL_CAP  = 12
FDH_COST      = 20000
FDH_CAP       = 512
OLT_COST      = 100000
OLT_CAP       = 4000
CABLE_PER_M   = 10
REUSE_FACTOR  = 0.25       # duct sharing: reused street sections cost this fraction
ROUNDS        = 3          # joint-feedback rounds (1 = greedy chain)
MAX_ADDRESSES = 0          # 0 = all 54,921; e.g. 10000 for a quick run
BRANCH = 'claude/stage-1-implementation-plan-9lv5f3'

In [ ]:
# --- machine specs (goes into the report) -----------------------------------
import subprocess
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
GPU  = sh('nvidia-smi -L') or '(no GPU / CPU runtime)'
CPU  = sh("lscpu | grep 'Model name' | sed 's/Model name: *//'")
CORES = sh('nproc')
RAM  = sh("free -g | awk '/Mem:/ {print $2}'") + ' GB'
MACHINE = f'{CPU} | {CORES} vCPU | {RAM} | {GPU}'
print(MACHINE)

In [ ]:
import os
if os.path.exists('/content/arcedge'):
    %cd /content/arcedge
    !git pull
else:
    token = ''  # <-- paste a GitHub token here if the repo is private
    url = f'https://{token}@github.com/fhk/arcedge.git' if token else 'https://github.com/fhk/arcedge.git'
    !git clone --branch {BRANCH} {url} /content/arcedge
    %cd /content/arcedge
!git log --oneline -1
!pip install -q pyyaml pyarrow shapely numpy
!cmake -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j$(nproc) 2>&1 | tail -1
!ctest --test-dir build --output-on-failure | tail -3

## Address data — upload `places_sf_04_2026.parquet`
(or skip to fall back to the committed 400-address downtown sample)

In [ ]:
import glob
PLACES = None
cands = glob.glob('/content/*.parquet') + glob.glob('/content/arcedge/*.parquet')
if not cands:
    try:
        from google.colab import files  # noqa
        print('Upload the places parquet now (cancel -> downtown sample):')
        up = files.upload()
        cands = ['/content/' + n for n in up]
    except Exception as e:
        print('upload skipped:', e)
PLACES = cands[0] if cands else None
print('places:', PLACES or 'downtown 400-address fallback')

In [ ]:
import yaml
if PLACES:
    addr_source = {'geoparquet': PLACES}
    if MAX_ADDRESSES:
        addr_source['max_points'] = MAX_ADDRESSES
    streets = 'data/sf_streets.graph'
else:
    addr_source = {'csv': 'examples/sf_dt_pois.csv'}
    streets = 'data/sf_downtown.graph'
model = {
  'model_version': 1, 'name': 'sf-joint-bench',
  'layers': {'streets': {'source': {'file': streets}},
             'addresses': {'source': addr_source, 'role': 'terminals'}},
  'couplings': [{'name': 'drops', 'from': 'addresses', 'to': 'streets',
                 'method': 'nearest_edge_split', 'snap_m': 0.5}],
  'facilities': {
    'terminal': {'open_cost': TERMINAL_COST, 'capacity': {'hard': TERMINAL_CAP}},
    'fdh':      {'open_cost': FDH_COST,      'capacity': {'hard': FDH_CAP}},
    'olt':      {'open_cost': OLT_COST,      'capacity': {'hard': OLT_CAP}}},
  'cable': {'fixed_cost_per_m': CABLE_PER_M, 'reuse_factor': REUSE_FACTOR},
  'commodities': [{'name': 'service',
                   'assignment': {'from': {'nodes': {'layer': 'addresses'}},
                                  'demand': 1, 'tier': 'terminal'}}],
}
os.makedirs('out', exist_ok=True)
yaml.safe_dump(model, open('out/joint_bench.yaml', 'w'), sort_keys=False)
print('config written')

## Run: compile once, solve the chain with feedback rounds

In [ ]:
import time
t0 = time.time()
!python3 scripts/arcedge_modelc.py out/joint_bench.yaml -o out/joint_bench \
    --solve --rounds {ROUNDS}
WALL_S = time.time() - t0
print(f'\ntotal wall-clock (compile + all rounds): {WALL_S:.0f} s')

## Performance report

In [ ]:
import json
s = json.load(open('out/joint_bench/tiers_summary.json'))
manifest = json.load(open('out/joint_bench/manifest.json'))
n_addr = manifest['report']['pois']
greedy = s['rounds'][0]['total_cost']

print(f"{'round':>5} {'true total':>12} {'vs greedy':>9} {'wall s':>7}  per-tier (facilities @ solver-s)")
for r in s['rounds']:
    tiers = '  '.join(f"{t['tier']}:{t['hubs']}@{t['ms']/1000:.0f}s" for t in r['tiers'])
    print(f"{r['round']:>5} {r['total_cost']:>12,.0f} {100*(r['total_cost']/greedy-1):>8.2f}% "
          f"{r['wall_s']:>7.0f}  {tiers}")
print(f"\nbest: {s['total_cost']:,.0f}  ({100*(1-s['total_cost']/greedy):.2f}% below greedy)"
      f"   cost/address: {s['total_cost']/n_addr:,.0f}   addresses: {n_addr}")

bench = dict(machine=MACHINE, addresses=n_addr, rounds=s['rounds'],
             best_total=s['total_cost'], greedy_total=greedy,
             wall_s_total=WALL_S,
             params=dict(terminal=[TERMINAL_COST, TERMINAL_CAP],
                         fdh=[FDH_COST, FDH_CAP], olt=[OLT_COST, OLT_CAP],
                         cable_per_m=CABLE_PER_M, rounds=ROUNDS))
json.dump(bench, open('joint_bench.json', 'w'), indent=2)
print('\nsaved joint_bench.json')

In [ ]:
# --- paste-ready report block ------------------------------------------------
print('--- arcedge joint-chain benchmark ---')
print(f'machine: {MACHINE}')
print(f'addresses: {n_addr}   rounds: {ROUNDS}   total wall: {WALL_S:.0f} s')
print()
print('| round | true total | vs greedy | wall s | terminals | fdh | olt |')
print('|---:|---:|---:|---:|---:|---:|---:|')
for r in s['rounds']:
    t = {x['tier']: x for x in r['tiers']}
    print(f"| {r['round']} | {r['total_cost']:,.0f} "
          f"| {100*(r['total_cost']/greedy-1):+.2f}% | {r['wall_s']:.0f} "
          f"| {t['terminal']['hubs']} | {t['fdh']['hubs']} | {t['olt']['hubs']} |")
best_total = s['total_cost']
print(f'best chain: {best_total:,.0f} ({100*(1-best_total/greedy):.2f}% below greedy)')

## Outputs: per-tier GeoParquet + bundle

In [ ]:
tier_pois = {1: 'out/joint_bench/tier0.pois', 2: 'out/joint_bench/tier1.pois',
             3: 'out/joint_bench/tier2.pois'}
for i, t in enumerate(s['tiers'], start=1):
    !python3 scripts/solution_to_geoparquet.py design \
        --graph out/joint_bench/access.graph --pois {tier_pois[i]} \
        --solution out/joint_bench/tier{i}.solution \
        --out out/joint_bench/tier{i}_{t['tier']}.parquet
!cp joint_bench.json /content/
!cd out/joint_bench && zip -q /content/joint_bench_output.zip *.parquet tiers_summary.json
!ls -la /content/joint_bench_output.zip /content/joint_bench.json
print('\nDownload from the Files sidebar; paste the report block back to record.')

## Notes

- `ROUNDS`: round 1 is the greedy chain; the best round can never be worse
  (the driver keeps the cheapest chain by true cost). Full SF typically
  converges by round 3–4.
- `MAX_ADDRESSES` subsamples for quick parameter sweeps.
- Solver threads default to all vCPUs — that's why the GPU-class runtime
  (more vCPUs) is faster here despite the GPU itself being idle.
- GPU kernel benchmarks (MCF batched SSSP): `arcedge_s2m1_benchmark.ipynb`.